In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [3]:
def report_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Report')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [4]:
report_deep_dive = report_research('2026-02-24', '2026-02-25')

In [5]:
report_deep_dive

[{'event_id': '1450_2026-02-23',
  'output': 'Anthropic said Chinese companies used its Claude model to help improve their own models (reporting of model distillation/use cases) — a noteworthy disclosure about model reuse and competitive implications for model IP, export controls, and model hardening efforts.',
  'sources': [{'url': 'https://www.reuters.com/world/china/chinese-companies-used-claude-improve-own-models-anthropic-says-2026-02-23/',
    'name': 'Reuters'}],
  'news_date': '2026-02-23',
  'topic': 'Report'},
 {'event_id': '1451_2026-02-23',
  'output': 'Anthropic published a technical/security post titled "Detecting and preventing distillation attacks" (Feb 23, 2026) describing identification and mitigation of a class of model-extraction/distillation attacks — a significant high-signal safety/security update from a major AI lab.',
  'sources': [{'url': 'https://www.anthropic.com/news/detecting-and-preventing-distillation-attacks',
    'name': 'Anthropic'}],
  'news_date': '

In [6]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              Report summary: {i['output']}
              Source: {i['sources']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [9]:
openai_research_workflow(report_deep_dive)

- REPORT SNAPSHOT - What was published, by whom, and when: Reuters reported on February 23, 2026 that Anthropic said three Chinese companies used its Claude model to illicitly obtain capabilities to improve their own models. [Source: Reuters, https://www.reuters.com/world/china/chinese-companies-used-claude-improve-own-models-anthropic-says-2026-02-23/]

- REPORT SNAPSHOT - Scope: geography, sectors, sample size, time window, method type: Geography — China; Sectors — frontier AI labs; Sample size — three labs (DeepSeek, Moonshot, MiniMax); Time window — campaigns active by February 23, 2026; Method type — distillation (model-extraction technique). [Source: Reuters, https://www.reuters.com/world/china/chinese-companies-used-claude-improve-own-models-anthropic-says-2026-02-23/]

- CORE FINDINGS - Quantified finding: total scale of activity: more than 16,000,000 interactions with Claude, using roughly 24,000 fake accounts. [Source: Reuters, https://www.reuters.com/world/china/chinese-comp

In [8]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()